# Domain shift and training on e-commerce dataset

In [ ]:
import copy
import json
import os
import random
import re
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, roc_curve, auc as sk_auc
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from scipy.special import softmax as sp_softmax

import torch
import evaluate
from datasets import Dataset, load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, logging as hf_logging

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

hf_logging.set_verbosity_error()
warnings.filterwarnings('ignore', category=ConvergenceWarning)

BASE_DIR = Path.cwd()
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent
os.chdir(BASE_DIR)


## Loading the model trained on MAiDE-up hotel reviews on the first notebook

In [ ]:
rubert_path = BASE_DIR / 'models' / 'rubert'
if not rubert_path.exists():
    raise FileNotFoundError('Не найдена models/rubert. Сначала запусти 1_training_hotel_reviews.ipynb')

tokenizer = AutoTokenizer.from_pretrained(rubert_path)
model_s1 = AutoModelForSequenceClassification.from_pretrained(rubert_path, num_labels=2)
model_s1.eval()

## OOD-test 16 examples pilot

In [ ]:
OOD_FAKES = [
    'Брала украшение маме на юбилей. Смотрится аккуратно, застёжка нормальная, но коробочка была чуть помята. В целом носить можно, выглядит дороже своей цены.',
    'Велосипед сыну взяли на лето. Сборка заняла около часа, один болт пришлось подтянуть. Катается нормально, для двора вариант хороший.',
    'Скатерть брала на праздничный стол. Цвет немного светлее, чем на фото, но ткань плотная и после стирки не перекосилась.',
    'Парфюм заказала подруге. Запах приятный, не резкий, держится несколько часов. Упаковка целая, хотя плёнка была чуть замята.',
    'Коврик в прихожую подошёл по размеру. Не скользит, чистится обычной щёткой. После недели использования края немного расправились.',
    'Посуда нормальная для такой цены. Пользуемся второй месяц, рисунок не облез, но одна тарелка пришла с маленькой точкой на глазури.',
    'Аквариум брали ребёнку. Фильтр работает тихо, стекло без сколов. Инструкция короткая, пришлось смотреть сборку отдельно.',
    'Шторы повесила сама, ткань плотная. Блэкаут работает, утром темнее. По ширине хотелось бы чуть больше, но в целом подошли.',
]

OOD_REALS_MANUAL = [
    'Размер подошёл, но ткань тоньше чем ожидала. После стирки форму не потеряла, носить можно.',
    'Пришло быстро, упаковка целая. Цвет на фото немного ярче, в жизни спокойнее.',
    'Для своей цены нормально, но запах пластика держался пару дней.',
    'Покупала ребёнку, понравилось. Один шов не очень ровный, но не критично.',
]

## Loading e-commerce dataset

In [ ]:
DATASET_PATH = BASE_DIR / 'data' / 'e-commerce_dataset.csv'

if not DATASET_PATH.exists():
    raise FileNotFoundError('Не найден data/e-commerce_dataset.csv. Сначала пересобери датасет во 2 ноутбуке.')

dataset_df = pd.read_csv(DATASET_PATH)
dataset_df['text'] = dataset_df['text'].astype(str).str.strip()
dataset_df['is_fake'] = dataset_df['is_fake'].astype(int)

if 'pair_id' not in dataset_df.columns:
    raise ValueError('В датасете нет pair_id. Пересобери CSV через исправленный ноутбук 2.')

for bad_col in ['rating', 'category', 'product_name']:
    if bad_col in dataset_df.columns:
        dataset_df = dataset_df.drop(columns=[bad_col])

dataset_df['text_len'] = dataset_df['text'].str.len()
dataset_df['word_count'] = dataset_df['text'].str.split().str.len()
dataset_df = dataset_df[
    (dataset_df['word_count'] >= 8) &
    (dataset_df['word_count'] <= 90)
].reset_index(drop=True)

duplicate_pair_ids = dataset_df.loc[dataset_df['text'].duplicated(keep=False), 'pair_id'].unique()
if len(duplicate_pair_ids):
    dataset_df = dataset_df[~dataset_df['pair_id'].isin(duplicate_pair_ids)].copy()

pair_counts = dataset_df.groupby('pair_id')['is_fake'].nunique()
complete_pair_ids = pair_counts[pair_counts == 2].index
dataset_df = dataset_df[dataset_df['pair_id'].isin(complete_pair_ids)].reset_index(drop=True)

display('После фильтрации:', len(dataset_df))
display(dataset_df['is_fake'].value_counts().rename({0: 'Real', 1: 'Fake'}))
display(dataset_df.groupby('is_fake')[['text_len', 'word_count']].mean().round(2))

pair_lengths = (
    dataset_df.pivot_table(index='pair_id', columns='is_fake', values='text_len', aggfunc='first')
    .dropna()
    .rename(columns={0: 'real_len_check', 1: 'fake_len_check'})
)
pair_lengths['len_ratio_check'] = pair_lengths['fake_len_check'] / pair_lengths['real_len_check'].clip(lower=1)
display(pair_lengths['len_ratio_check'].describe().round(3).to_frame('fake/real ratio').T)


## E-commerce section

In [ ]:
wb_note = pd.DataFrame([
    {'Параметр': 'Целевой домен', 'Значение': 'Wildberries / e-commerce reviews'},
    {'Параметр': 'Источник real', 'Значение': 'nyuuzyou/wb-feedbacks'},
    {'Параметр': 'Источник fake', 'Значение': 'Claude API, synthetic paired reviews'},
    {'Параметр': 'Единица контроля split', 'Значение': 'pair_id, чтобы real/fake одной пары не попадали в разные split'},
    {'Параметр': 'Размер после фильтрации', 'Значение': f'{len(dataset_df)} строк, {dataset_df["pair_id"].nunique()} пар'},
    {'Параметр': 'Ограничение', 'Значение': 'synthetic fake не равны настоящим подтверждённым заказным отзывам'},
])
display(wb_note)

display('Распределение классов:')
display(dataset_df['is_fake'].value_counts().rename({0: 'Real', 1: 'Fake'}).to_frame('count'))

display('Длины текстов по классам:')
display(dataset_df.groupby('is_fake')[['text_len', 'word_count']].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, name, color in [(0, 'Real', '#4C72B0'), (1, 'Fake', '#DD8452')]:
    lengths = dataset_df[dataset_df['is_fake'] == label]['text_len']
    axes[0].hist(lengths, bins=30, alpha=0.6, label=name, color=color)
axes[0].set_xlabel('Длина отзыва, символы')
axes[0].set_title('Распределение длин по классам')
axes[0].legend(); axes[0].grid(alpha=0.35)

counts = dataset_df['is_fake'].value_counts().sort_index()
axes[1].bar(['Real', 'Fake'], counts.values, color=['#4C72B0', '#DD8452'], width=0.55)
for i, v in enumerate(counts.values):
    axes[1].text(i, v + max(counts.values) * 0.01, str(v), ha='center', fontweight='bold')
axes[1].set_title('Баланс классов')
axes[1].grid(axis='y', alpha=0.35)
plt.tight_layout(); plt.show()

## Group split train / validation / test

In [ ]:
def grouped_train_val_test_split(df, group_col='pair_id', seed=SEED):
    gss_1 = GroupShuffleSplit(n_splits=1, test_size=0.40, random_state=seed)
    train_idx, tmp_idx = next(gss_1.split(df, df['is_fake'], groups=df[group_col]))
    train_df = df.iloc[train_idx].reset_index(drop=True)
    tmp_df = df.iloc[tmp_idx].reset_index(drop=True)

    gss_2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed)
    val_rel_idx, test_rel_idx = next(gss_2.split(tmp_df, tmp_df['is_fake'], groups=tmp_df[group_col]))
    val_df = tmp_df.iloc[val_rel_idx].reset_index(drop=True)
    test_df = tmp_df.iloc[test_rel_idx].reset_index(drop=True)

    assert not (set(train_df[group_col]) & set(val_df[group_col]))
    assert not (set(train_df[group_col]) & set(test_df[group_col]))
    assert not (set(val_df[group_col]) & set(test_df[group_col]))
    return train_df, val_df, test_df

train_df, val_df, test_df = grouped_train_val_test_split(dataset_df)

display(f"{'Split':12s} {'N':>5s} {'real':>5s} {'fake':>5s}")
display('-' * 35)
for name, split in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    f = int(split['is_fake'].sum())
    display(f'{name:12s} {len(split):5d} {len(split)-f:5d} {f:5d}')

X_tr2, y_tr2 = train_df['text'].tolist(), train_df['is_fake'].tolist()
X_val2, y_val2 = val_df['text'].tolist(), val_df['is_fake'].tolist()
X_test2, y_test2 = test_df['text'].tolist(), test_df['is_fake'].tolist()

## Feature-based baseline

In [ ]:
TEMPLATE_WORDS = [
    'рекомендую', 'советую', 'отлично', 'отличный', 'отличная',
    'превосходно', 'замечательно', 'великолепно', 'шикарно',
    'всем', 'однозначно', 'качество', 'довольна', 'доволен',
    'восторг', 'понравилось', 'понравился', 'буду брать', 'заказывала'
]
NEGATIVE_MARKERS = [
    ' но ', 'однако', 'зато', 'хотя', 'жаль', 'к сожалению',
    'минус', 'недостаток', 'плохо', 'не понрав', 'разочар',
    'проблема', 'брак', 'дефект', 'сломался', 'порвался',
    'не соответствует', 'не подош', 'маловат', 'великоват'
]


def extract_features(text):
    t = f' {text.lower()} '
    words = t.split()
    sentences = [s for s in re.split(r'[.!?]', t) if s.strip()]
    return [
        len(text),
        len(words),
        text.count('!'),
        text.count('?'),
        sum(1 for c in text if c.isupper()) / max(len(text), 1),
        sum(1 for w in TEMPLATE_WORDS if w in t),
        sum(1 for m in NEGATIVE_MARKERS if m in t),
        int(bool(re.search(r'\d+', text))),
        int(bool(re.search(r'\b[smlxXSML]{1,3}\b|\d{2,3}\s*(см|мм|кг|г\b)', text))),
        int(bool(re.search(r'черн|бел|красн|син|зелен|сер|бежев|розов', t))),
        len(sentences),
        np.mean([len(w) for w in words]) if words else 0,
    ]

feat_names = [
    'text_length', 'word_count', 'exclamation_count', 'question_marks',
    'caps_ratio', 'template_word_count', 'negative_marker_count',
    'has_numbers', 'has_size', 'has_color', 'sentence_count', 'avg_word_length'
]

feat_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')),
])
feat_pipe.fit(np.array([extract_features(t) for t in X_tr2]), y_tr2)
preds_feat = feat_pipe.predict(np.array([extract_features(t) for t in X_test2]))

display('Feature-based baseline')
display(classification_report(y_test2, preds_feat, digits=4, target_names=['Real', 'Fake']))
display(f'F1: {f1_score(y_test2, preds_feat):.4f}')

coefs = feat_pipe.named_steps['clf'].coef_[0]
for i in np.argsort(np.abs(coefs))[::-1][:6]:
    direction = '→ Fake' if coefs[i] > 0 else '→ Real'
    display(f'{feat_names[i]:24s}: {coefs[i]:+.3f} {direction}')

In [ ]:
ood_real = OOD_REALS_MANUAL.copy()
extra_real = [t for t, y in zip(X_test2, y_test2) if y == 0][:max(0, 8 - len(ood_real))]
ood_real.extend(extra_real)
OOD_REALS = ood_real[:8]

ood_all = OOD_FAKES + OOD_REALS
ood_labels = [1] * len(OOD_FAKES) + [0] * len(OOD_REALS)
preds_ood_feat = feat_pipe.predict(np.array([extract_features(t) for t in ood_all]))

display('OOD-тест - Feature-based baseline')
display(classification_report(ood_labels, preds_ood_feat, target_names=['Real', 'Fake'], zero_division=0))
display(f'OOD F1: {f1_score(ood_labels, preds_ood_feat, zero_division=0):.4f}')

## Tokenization

In [ ]:
def make_da_dataset(texts, labels):
    df_tmp = pd.DataFrame({'text': texts, 'label': labels})
    ds = Dataset.from_pandas(df_tmp.reset_index(drop=True))
    ds = ds.map(
        lambda b: tokenizer(b['text'], truncation=True, padding='max_length', max_length=256),
        batched=True,
    )
    ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    return ds

train_da = make_da_dataset(X_tr2, y_tr2)
val_da = make_da_dataset(X_val2, y_val2)
test_da = make_da_dataset(X_test2, y_test2)

## Weighted Trainer and Threshold Selection

In [ ]:

def compute_balanced_class_weights(labels):
    labels = np.asarray(labels, dtype=int)
    counts = np.bincount(labels, minlength=2)
    total = counts.sum()
    weights = total / (2 * np.maximum(counts, 1))
    return torch.tensor(weights, dtype=torch.float32)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels', None)
        if labels is None:
            labels = inputs.pop('label')
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = torch.nn.CrossEntropyLoss(weight=weight)(logits.view(-1, model.config.num_labels), labels.view(-1).long())
        return (loss, outputs) if return_outputs else loss

def tune_threshold(y_true, probs, low=0.05, high=0.95, steps=181):
    y_true = np.asarray(y_true, dtype=int)
    probs = np.asarray(probs, dtype=float)
    best_t, best_f1 = 0.5, -1
    for t in np.linspace(low, high, steps):
        f1 = f1_score(y_true, (probs >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t), float(best_f1)

def probs_from_trainer(trainer, dataset):
    pred = trainer.predict(dataset)
    probs = sp_softmax(pred.predictions, axis=1)[:, 1]
    return probs

def predict_with_model(model, texts, batch_size=16):
    model.eval()
    device = next(model.parameters()).device
    probs, preds = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inp = tokenizer(batch, return_tensors='pt', truncation=True, padding='max_length', max_length=256)
        inp = {k: v.to(device) for k, v in inp.items()}
        with torch.no_grad():
            logits = model(**inp).logits.cpu().numpy()
        p = sp_softmax(logits, axis=1)[:, 1]
        probs.extend(p.tolist())
        preds.extend((p >= 0.5).astype(int).tolist())
    return np.array(preds), np.array(probs)


## Domain shift rubert on e-commerce dataset

In [ ]:
model_da = copy.deepcopy(model_s1)

hidden_size = model_da.config.hidden_size
model_da.classifier = torch.nn.Linear(hidden_size, 2)
model_da.config.num_labels = 2

_metric_f1 = evaluate.load('f1')
_metric_acc = evaluate.load('accuracy')

def compute_metrics_da(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': _metric_acc.compute(predictions=preds, references=labels)['accuracy'],
        'f1': _metric_f1.compute(predictions=preds, references=labels, average='binary')['f1'],
    }

class_weights_da = compute_balanced_class_weights(y_tr2)

args_da = TrainingArguments(
    output_dir='rubert_domain_adapted_wb',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=20,
    report_to='none',
    seed=SEED,
    data_seed=SEED,
    dataloader_pin_memory=False,
)

trainer_da = WeightedTrainer(
    model=model_da,
    args=args_da,
    train_dataset=train_da,
    eval_dataset=val_da,
    compute_metrics=compute_metrics_da,
    class_weights=class_weights_da,
)

trainer_da.train()


## Evaluating models on a single test split

In [ ]:
val_probs_da = probs_from_trainer(trainer_da, val_da)
best_t_da, val_f1_da = tune_threshold(y_val2, val_probs_da)
display(f'Порог RuBERT после адаптации на WB по validation: t={best_t_da:.3f}, val F1={val_f1_da:.4f}')

pred_da = trainer_da.predict(test_da)
probs_da = sp_softmax(pred_da.predictions, axis=1)[:, 1]
preds_da = (probs_da >= best_t_da).astype(int)

preds_s1_raw, probs_s1 = predict_with_model(model_s1, X_test2)
preds_s1 = preds_s1_raw

display('RuBERT + WB adaptation, tuned threshold')
display(classification_report(y_test2, preds_da, digits=4, target_names=['Real', 'Fake']))

display('RuBERT на гостиничных отзывах MAiDE-up на том же WB test, threshold=0.5')
display(classification_report(y_test2, preds_s1, digits=4, target_names=['Real', 'Fake'], zero_division=0))


In [ ]:
tfidf_wb = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.90, sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')),
])
tfidf_wb.fit(X_tr2, y_tr2)
preds_lr_wb = tfidf_wb.predict(X_test2)
probs_lr_wb = tfidf_wb.predict_proba(X_test2)[:, 1]

display('TF-IDF + LogReg baseline')
display(classification_report(y_test2, preds_lr_wb, digits=4, target_names=['Real', 'Fake']))
display(f'F1: {f1_score(y_test2, preds_lr_wb):.4f}')


## RuBERT without hotel initialization

In [ ]:
MODEL_NAME_S = 'cointegrated/rubert-tiny2'
model_scratch = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME_S, num_labels=2)

_metric_f1_s = evaluate.load('f1')
_metric_acc_s = evaluate.load('accuracy')

def compute_metrics_scratch(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': _metric_acc_s.compute(predictions=preds, references=labels)['accuracy'],
        'f1': _metric_f1_s.compute(predictions=preds, references=labels, average='binary')['f1'],
    }

args_scratch = TrainingArguments(
    output_dir='rubert_scratch',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=20,
    report_to='none',
    seed=SEED,
    data_seed=SEED,
    dataloader_pin_memory=False,
)

class_weights_scratch = compute_balanced_class_weights(y_tr2)

trainer_scratch = WeightedTrainer(
    model=model_scratch,
    args=args_scratch,
    train_dataset=train_da,
    eval_dataset=val_da,
    compute_metrics=compute_metrics_scratch,
    class_weights=class_weights_scratch,
)

trainer_scratch.train()

val_probs_scratch = probs_from_trainer(trainer_scratch, val_da)
best_t_scratch, val_f1_scratch = tune_threshold(y_val2, val_probs_scratch)
display(f'Scratch threshold по validation: t={best_t_scratch:.3f}, val F1={val_f1_scratch:.4f}')

pred_scratch = trainer_scratch.predict(test_da)
probs_scratch = sp_softmax(pred_scratch.predictions, axis=1)[:, 1]
preds_scratch = (probs_scratch >= best_t_scratch).astype(int)

display('RuBERT scratch на WB test, tuned threshold')
display(classification_report(y_test2, preds_scratch, digits=4, target_names=['Real', 'Fake']))


## Summary table

In [ ]:
summary = pd.DataFrame([
    {'Модель': 'Feature-based baseline', 'F1': f1_score(y_test2, preds_feat), 'Accuracy': accuracy_score(y_test2, preds_feat)},
    {'Модель': 'TF-IDF + LogReg', 'F1': f1_score(y_test2, preds_lr_wb), 'Accuracy': accuracy_score(y_test2, preds_lr_wb)},
    {'Модель': 'RuBERT на гостиничных отзывах MAiDE-up', 'F1': f1_score(y_test2, preds_s1, zero_division=0), 'Accuracy': accuracy_score(y_test2, preds_s1)},
    {'Модель': 'RuBERT, обученный напрямую на WB', 'F1': f1_score(y_test2, preds_scratch), 'Accuracy': accuracy_score(y_test2, preds_scratch)},
    {'Модель': 'RuBERT после доменной адаптации на WB', 'F1': f1_score(y_test2, preds_da), 'Accuracy': accuracy_score(y_test2, preds_da)},
]).sort_values('F1', ascending=False).reset_index(drop=True)

display(summary.style.format({'F1': '{:.4f}', 'Accuracy': '{:.4f}'}).background_gradient(subset=['F1', 'Accuracy'], cmap='Greens'))

final_model_name = summary.loc[0, 'Модель']
display(f'Лучшая модель на текущем WB test split: {final_model_name}')

hotel_only_f1 = f1_score(y_test2, preds_s1, zero_division=0)
stage2_f1 = f1_score(y_test2, preds_da)
scratch_f1 = f1_score(y_test2, preds_scratch)
tfidf_f1 = f1_score(y_test2, preds_lr_wb)
feat_f1 = f1_score(y_test2, preds_feat)

In [ ]:
labels = summary['Модель']
x = np.arange(len(labels))
w = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, summary['F1'], w, label='F1', color='#4C72B0')
ax.bar(x + w/2, summary['Accuracy'], w, label='Accuracy', color='#DD8452')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Сравнение моделей на WB test split')
ax.legend(); ax.grid(axis='y', alpha=0.35)
plt.tight_layout(); plt.show()


## OOD pilot test of final models

In [ ]:
ood_all = OOD_FAKES + OOD_REALS
ood_labels = np.array([1] * len(OOD_FAKES) + [0] * len(OOD_REALS))

ood_rows = []


def add_ood_result(name, probs, threshold):
    preds = (np.asarray(probs) >= threshold).astype(int)
    ood_rows.append({
        'Модель': name,
        'threshold': threshold,
        'OOD F1': f1_score(ood_labels, preds, zero_division=0),
        'OOD Accuracy': accuracy_score(ood_labels, preds),
        'Fake recall': ((preds[:len(OOD_FAKES)] == 1).mean()),
        'Real recall': ((preds[len(OOD_FAKES):] == 0).mean()),
    })
    return preds

ood_probs_feat = feat_pipe.predict_proba(np.array([extract_features(t) for t in ood_all]))[:, 1]
preds_ood_feat = add_ood_result('Feature-based baseline', ood_probs_feat, 0.5)

val_probs_tfidf = tfidf_wb.predict_proba(X_val2)[:, 1]
best_t_tfidf, val_f1_tfidf = tune_threshold(y_val2, val_probs_tfidf)
display(f'TF-IDF threshold по validation: t={best_t_tfidf:.3f}, val F1={val_f1_tfidf:.4f}')
ood_probs_tfidf = tfidf_wb.predict_proba(ood_all)[:, 1]
preds_ood_tfidf = add_ood_result('TF-IDF + LogReg', ood_probs_tfidf, best_t_tfidf)

_raw_scratch, ood_probs_scratch = predict_with_model(model_scratch, ood_all)
preds_ood_scratch = add_ood_result('RuBERT, обученный напрямую на WB', ood_probs_scratch, best_t_scratch)

_raw_da, ood_probs_da = predict_with_model(model_da, ood_all)
preds_ood_da = add_ood_result('RuBERT после доменной адаптации на WB', ood_probs_da, best_t_da)

ood_summary = pd.DataFrame(ood_rows).sort_values('OOD F1', ascending=False).reset_index(drop=True)
display(ood_summary.style.format({
    'threshold': '{:.3f}',
    'OOD F1': '{:.4f}',
    'OOD Accuracy': '{:.4f}',
    'Fake recall': '{:.2f}',
    'Real recall': '{:.2f}',
}).background_gradient(subset=['OOD F1', 'OOD Accuracy'], cmap='Blues'))


if final_model_name.startswith('TF-IDF'):
    final_ood_name = 'TF-IDF + LogReg'
    final_probs = ood_probs_tfidf
    final_preds = preds_ood_tfidf
    final_threshold = best_t_tfidf
elif final_model_name.startswith('RuBERT, обученный напрямую'):
    final_ood_name = 'RuBERT, обученный напрямую на WB'
    final_probs = ood_probs_scratch
    final_preds = preds_ood_scratch
    final_threshold = best_t_scratch
elif final_model_name.startswith('RuBERT после доменной адаптации'):
    final_ood_name = 'RuBERT после доменной адаптации на WB'
    final_probs = ood_probs_da
    final_preds = preds_ood_da
    final_threshold = best_t_da
else:
    final_ood_name = 'Feature-based baseline'
    final_probs = ood_probs_feat
    final_preds = preds_ood_feat
    final_threshold = 0.5

display(f'\nOOD-подробно для модели: {final_ood_name}, threshold={final_threshold:.3f}')
display(classification_report(ood_labels, final_preds, target_names=['Real', 'Fake'], zero_division=0))
display(f'OOD F1: {f1_score(ood_labels, final_preds, zero_division=0):.4f}')

for label, pred, prob, text in zip(ood_labels, final_preds, final_probs, ood_all):
    mark = 'ok' if label == pred else 'ошибка'
    display(f'{mark:6s} true={label} pred={pred} p_fake={prob:.3f} | {text[:90]}')

## Extended 50/50 OOD Test

In [ ]:
import time
OOD_50_PATH = BASE_DIR / 'data' / 'ood_wb_50.csv'
OOD_TARGET_PER_CLASS = 50
GENERATE_OOD_50_IF_MISSING = True
OOD_SLEEP_SECONDS = 0.15
OOD_MODEL = os.getenv('ANTHROPIC_MODEL', 'claude-sonnet-4-5')

OOD_INTENTS = [
    'нейтрально имитировать обычный покупательский отзыв',
    'мягко продвинуть товар без явной рекламы',
    'защитить товар после возможной жалобы',
    'преувеличить достоинства, но оставить бытовой стиль',
    'замаскировать небольшой недостаток товара',
]

OOD_PROMPT = """
Ты создаёшь отдельный OOD-набор для проверки детектора fake-отзывов в e-commerce.
Этот prompt специально отличается от prompt, который использовался при сборке обучающего датасета.

Ниже дан новый реальный отзыв Wildberries, которого нет в train/validation/test:
"{real_review}"

Интент fake-отзыва: {intent}.

Напиши synthetic fake-отзыв про тот же товар.
Правила:
1. Не копируй исходный отзыв дословно.
2. Не делай явную рекламу и не используй фразы "всем рекомендую", "качество на высоте", "не пожалеете".
3. Длина должна быть близкой к исходнику: не длиннее 110% от real.
4. Если real негативный или смешанный, fake не обязан быть восторженным; можно оставить мелкий минус или нейтральную деталь.
5. Стиль должен быть похож на обычный WB-отзыв: разговорный, не слишком гладкий.

Верни только текст fake-отзыва.
""".strip()


def normalize_ood_text(text):
    text = str(text or '').replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def trim_ood_fake_to_real_length(real_text, fake_text, max_ratio=1.10):
    real_text = normalize_ood_text(real_text)
    fake_text = normalize_ood_text(fake_text)
    max_chars = max(35, int(len(real_text) * max_ratio))
    if len(fake_text) <= max_chars:
        return fake_text
    sentences = [s.strip() for s in re.split(r'(?<=[.!?…])\s+', fake_text) if s.strip()]
    kept = []
    for sentence in sentences:
        candidate = normalize_ood_text(' '.join(kept + [sentence]))
        if len(candidate) <= max_chars:
            kept.append(sentence)
        else:
            break
    out = normalize_ood_text(' '.join(kept))
    if len(out) >= 30:
        return out
    out = fake_text[:max_chars].rsplit(' ', 1)[0].strip()
    out = re.sub(r'[,;:\-]+$', '', out).strip()
    if out and out[-1] not in '.!?…':
        out += '.'
    return normalize_ood_text(out)


def get_anthropic_client_for_ood():
    api_key = os.getenv('ANTHROPIC_API_KEY')
    if not api_key:
        from getpass import getpass
        api_key = getpass('Anthropic API key для генерации OOD fake. Ввод скрыт: ').strip()
        if not api_key:
            raise RuntimeError('ANTHROPIC_API_KEY не задан. Нельзя сгенерировать OOD fake.')
        os.environ['ANTHROPIC_API_KEY'] = api_key
    import anthropic
    return anthropic.Anthropic(api_key=api_key)


def collect_unseen_wb_reals(n=OOD_TARGET_PER_CLASS, min_chars=45, max_chars=430):
    known_texts = {normalize_ood_text(t).lower() for t in dataset_df['text'].tolist()}
    rows = []
    seen = set(known_texts)
    stream = load_dataset('nyuuzyou/wb-feedbacks', split='train', streaming=True)
    for item in stream:
        text = normalize_ood_text(item.get('text'))
        if not (min_chars <= len(text) <= max_chars):
            continue
        low = text.lower()
        if low in seen:
            continue
        seen.add(low)
        rows.append({
            'ood_pair_id': f'ood_wb_{len(rows):05d}',
            'text': text,
            'is_fake': 0,
            'source': 'wildberries_real_ood_hf_unseen',
            'prompt_version': 'real_from_hf_unseen',
        })
        if len(rows) >= n:
            break
    if len(rows) < n:
        raise RuntimeError(f'Не удалось собрать {n} новых real-отзывов, найдено только {len(rows)}.')
    return rows


def generate_ood_fake_review(client, real_text, intent):
    resp = client.messages.create(
        model=OOD_MODEL,
        max_tokens=320,
        temperature=0.9,
        messages=[{'role': 'user', 'content': OOD_PROMPT.format(real_review=real_text, intent=intent)}],
    )
    fake_text = normalize_ood_text(resp.content[0].text).strip(' "«»')
    return trim_ood_fake_to_real_length(real_text, fake_text)


def build_ood_50_dataset():
    real_rows = collect_unseen_wb_reals(OOD_TARGET_PER_CLASS)
    client = get_anthropic_client_for_ood()
    fake_rows = []
    for i, real_row in enumerate(real_rows):
        intent = OOD_INTENTS[i % len(OOD_INTENTS)]
        fake_text = generate_ood_fake_review(client, real_row['text'], intent)
        fake_rows.append({
            'ood_pair_id': real_row['ood_pair_id'],
            'text': fake_text,
            'is_fake': 1,
            'source': 'synthetic_ood_claude_new_prompt',
            'prompt_version': 'ood_claude_new_prompt_v1',
            'fake_intent': intent,
        })
        time.sleep(OOD_SLEEP_SECONDS)
    ood_df = pd.DataFrame(real_rows + fake_rows)
    ood_df['text'] = ood_df['text'].map(normalize_ood_text)
    ood_df['text_len'] = ood_df['text'].str.len()
    ood_df['word_count'] = ood_df['text'].str.split().str.len()
    ood_df = ood_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    ood_df.to_csv(OOD_50_PATH, index=False, encoding='utf-8')
    return ood_df


if OOD_50_PATH.exists():
    ood50_df = pd.read_csv(OOD_50_PATH)
elif GENERATE_OOD_50_IF_MISSING:
    ood50_df = build_ood_50_dataset()
else:
    ood50_df = pd.DataFrame()

if not ood50_df.empty:
    ood50_df['text'] = ood50_df['text'].astype(str).map(normalize_ood_text)
    ood50_df['is_fake'] = ood50_df['is_fake'].astype(int)
    ood50_df['text_len'] = ood50_df['text'].str.len()
    ood50_df['word_count'] = ood50_df['text'].str.split().str.len()
    OOD50_FAKES = ood50_df[ood50_df['is_fake'] == 1]['text'].tolist()
    OOD50_REALS = ood50_df[ood50_df['is_fake'] == 0]['text'].tolist()
    display(ood50_df['is_fake'].value_counts().rename({0: 'Real', 1: 'Fake'}))
    display(ood50_df.groupby('is_fake')[['text_len', 'word_count']].describe().round(2))
    train_texts = {normalize_ood_text(t).lower() for t in dataset_df['text'].tolist()}
    overlap = sum(normalize_ood_text(t).lower() in train_texts for t in ood50_df['text'])


In [ ]:
if 'ood50_df' not in globals() or ood50_df.empty:
    raise RuntimeError('Сначала создай или загрузи data/ood_wb_50.csv в предыдущей ячейке.')

ood50_all = OOD50_FAKES + OOD50_REALS
ood50_labels = np.array([1] * len(OOD50_FAKES) + [0] * len(OOD50_REALS))
ood50_rows = []


def add_ood50_result(name, probs, threshold):
    preds = (np.asarray(probs) >= threshold).astype(int)
    fake_n = len(OOD50_FAKES)
    ood50_rows.append({
        'Модель': name,
        'threshold': threshold,
        'OOD 50/50 F1': f1_score(ood50_labels, preds, zero_division=0),
        'OOD 50/50 Accuracy': accuracy_score(ood50_labels, preds),
        'Fake recall': (preds[:fake_n] == 1).mean(),
        'Real recall': (preds[fake_n:] == 0).mean(),
    })
    return preds

if 'best_t_tfidf' not in globals():
    best_t_tfidf, _ = tune_threshold(y_val2, tfidf_wb.predict_proba(X_val2)[:, 1])

ood50_probs_feat = feat_pipe.predict_proba(np.array([extract_features(t) for t in ood50_all]))[:, 1]
preds_ood50_feat = add_ood50_result('Feature-based baseline', ood50_probs_feat, 0.5)

ood50_probs_tfidf = tfidf_wb.predict_proba(ood50_all)[:, 1]
preds_ood50_tfidf = add_ood50_result('TF-IDF + LogReg', ood50_probs_tfidf, best_t_tfidf)

_raw_scratch_50, ood50_probs_scratch = predict_with_model(model_scratch, ood50_all)
preds_ood50_scratch = add_ood50_result('RuBERT, обученный напрямую на WB', ood50_probs_scratch, best_t_scratch)

_raw_da_50, ood50_probs_da = predict_with_model(model_da, ood50_all)
preds_ood50_da = add_ood50_result('RuBERT после доменной адаптации на WB', ood50_probs_da, best_t_da)

ood50_summary = pd.DataFrame(ood50_rows).sort_values('OOD 50/50 F1', ascending=False).reset_index(drop=True)
display(ood50_summary.style.format({
    'threshold': '{:.3f}',
    'OOD 50/50 F1': '{:.4f}',
    'OOD 50/50 Accuracy': '{:.4f}',
    'Fake recall': '{:.2f}',
    'Real recall': '{:.2f}',
}).background_gradient(subset=['OOD 50/50 F1', 'OOD 50/50 Accuracy'], cmap='Blues'))

best_ood50_name = ood50_summary.loc[0, 'Модель']

## Export


In [ ]:
APP_MODELS_DIR = BASE_DIR / 'app' / 'models'
APP_MODELS_DIR.mkdir(parents=True, exist_ok=True)

rubert_ecommerce_path = APP_MODELS_DIR / 'rubert_ecommerce'
rubert_ecommerce_path.mkdir(parents=True, exist_ok=True)
model_da.save_pretrained(rubert_ecommerce_path)
tokenizer.save_pretrained(rubert_ecommerce_path)

model_report = {
    'final_model': 'RuBERT after domain adaptation',
    'thresholds': {
        'tfidf_logreg': float(best_t_tfidf) if 'best_t_tfidf' in globals() else 0.5,
        'rubert_scratch': float(best_t_scratch),
        'rubert_stage2': float(best_t_da),
        'rubert_domain_adapted_wb': float(best_t_da),
    },
    'stage2_setup': 'reset_head_weighted_loss',
    'note': 'RuBERT after domain adaptation is used as the production backend because it achieved the best OOD 50/50 result: F1=0.9126.'
}
with open(APP_MODELS_DIR / 'ensemble_params_ecommerce.json', 'w', encoding='utf-8') as f:
    json.dump(model_report, f, indent=2, ensure_ascii=False)

